In [1]:
import sys
import os
from pathlib import Path
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from config.settings import Config
from src.data_ingestion.extractorLoad import BatchExtractLoad
from src.data_processing.data_transformation import DataTransformer
from src.data_storage.data_lake import DataLakeManager
from src.data_quality.data_quality import DataQuality


spark = SparkSession.builder \
    .appName(Config.SPARK_APP_NAME) \
    .master(Config.SPARK_MASTER) \
    .config("spark.executor.memory", Config.SPARK_EXECUTOR_MEMORY) \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1,org.postgresql:postgresql:42.7.3") \
    .config("spark.pyspark.python", "/opt/homebrew/bin/python3.9") \
    .config("spark.pyspark.driver.python", "/opt/homebrew/bin/python3.9") \
    .getOrCreate()

26/01/22 17:44:36 WARN Utils: Your hostname, MacBook-Pro-de-Eduardo.local resolves to a loopback address: 127.0.0.1; using 192.168.3.96 instead (on interface en0)
26/01/22 17:44:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/eduardoalberto/.ivy2/cache
The jars for the packages stored in: /Users/eduardoalberto/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ad0b73bb-805c-4dfb-8681-6e879e5196e5;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central


:: loading settings :: url = jar:file:/Users/eduardoalberto/opt/spark-3.5.4/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 1717ms :: artifacts dl 2ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   5   |   1   |   0   |   0   

In [7]:


def from_database(query: str) -> DataFrame:
        return spark.read \
            .format("jdbc") \
            .option("url", "jdbc:postgresql://localhost:5432/dbpostgres")\
            .option("query", query) \
            .option("user", os.getenv("DB_USER", "postgres")) \
            .option("password", os.getenv("DB_PASSWORD", "postgre123")) \
            .option("driver", "org.postgresql.Driver") \
            .load()

df = from_database("SELECT * FROM tb_netflix")

df.show()

+---+-------------------+------------------+---------+------------+---------------+-----------------+--------------------+
| id|              filme|        ator_atriz|ano_filme|  bilheteria|tipo_plataforma|       tipo_filme|            dt_carga|
+---+-------------------+------------------+---------+------------+---------------+-----------------+--------------------+
|  1|Aventuras no Espaço|       Chris Pratt|     2017|523000000.00|      Streaming|             Ação|2025-11-13 02:05:...|
|  2| O Mistério da Casa|       Emily Blunt|     2019|342000000.00|         Cinema|         Suspense|2025-11-13 02:05:...|
|  3|       Ritmo Urbano|        Will Smith|     2020|455000000.00|      Streaming|            Drama|2025-11-13 02:05:...|
|  4|    Noite em Tóquio|Scarlett Johansson|     2018|389000000.00|         Cinema|          Romance|2025-11-13 02:05:...|
|  5|   Planeta Selvagem|       Zoe Saldana|     2021|612000000.00|      Streaming|Ficção Científica|2025-11-13 02:05:...|
|  6| Sombras do